In [1]:
import pandas as pd
import numpy as np
import os
import gc
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score

BASE_DIR = ".."
DATA_PATH = os.path.join(BASE_DIR, "data")
SUB_PATH = os.path.join(BASE_DIR, "submissions")

def reduce_mem(df):
    for col in df.columns:
        if df[col].dtype == 'float64':
            df[col] = df[col].astype('float32')
        if df[col].dtype == 'int64':
            df[col] = df[col].astype('int32')
    return df

print("Cell 1 Aman!")

Cell 1 Aman!


In [2]:
df_train = pd.read_csv(os.path.join(DATA_PATH, "train.csv"))
df_test = pd.read_csv(os.path.join(DATA_PATH, "test.csv"))

def clean_and_align(df):
    # Bersihkan Battery_Level (Hanya ambil angka)
    df['Battery_Level'] = df['Battery_Level'].astype(str).str.extract('(\d+)').astype(float)
    
    # Handle Signal_Strength yang isinya Edge/4G/5G
    # Kita buat kategori sederhana agar Train & Test sinkron
    df['Signal_Strength'] = df['Signal_Strength'].astype(str).apply(lambda x: 'Low' if x in ['Edge', '3G'] else 'High')
    
    return reduce_mem(df)

df_train = clean_and_align(df_train)
df_test = clean_and_align(df_test)
print("Cell 2: Data Sinkron!")

Cell 2: Data Sinkron!


In [3]:
def build_final_features(df):
    df['Timestamp'] = pd.to_datetime(df['Timestamp'])
    df['hour'] = df['Timestamp'].dt.hour
    
    # 1. Anomali Sensor
    df['accel_mag'] = np.sqrt(df['Accel_X']**2 + df['Accel_Y']**2 + df['Accel_Z']**2)
    df['gyro_abs'] = df['Gyro_Z'].abs()
    
    # 2. Rasio Jarak (Sinyal Utama Navigation Issue)
    lat_diff = (df['Dropoff_Lat'] - df['Pickup_Lat']).abs()
    long_diff = (df['Dropoff_Long'] - df['Pickup_Long']).abs()
    euclidean = np.sqrt(lat_diff**2 + long_diff**2)
    # Efisiensi: Semakin besar angkanya, semakin muter-muter jalannya
    df['dist_efficiency'] = df['Distance_KM'] / (euclidean + 0.001)
    
    # 3. Anomali Harga vs Jarak (Sinyal Fraud)
    df['price_per_km'] = df['Est_Price_IDR'] / (df['Distance_KM'] + 0.5)
    
    # 4. Device Brand
    df['Brand'] = df['Device_FP'].astype(str).apply(lambda x: x.split('-')[0])
    
    # 5. Null Count sebagai sinyal teknis
    df['missing_info'] = df.isnull().sum(axis=1)

    drop_cols = ['Timestamp', 'Trip_ID'] if 'Trip_ID' in df.columns else ['Timestamp']
    return df.drop(columns=drop_cols)

y = df_train['Trip_Label']
X = build_final_features(df_train.drop(columns=['Trip_Label']))
X_test_final = build_final_features(df_test)
print("Cell 3: Fitur 0.6+ Selesai!")

Cell 3: Fitur 0.6+ Selesai!


In [4]:
# 1. Ambil sampel awal 10% (800rb baris) - Sangat aman buat RAM 16GB
X_samp, _, y_samp, _ = train_test_split(X, y, train_size=0.10, stratify=y, random_state=42)

# 2. Split jadi Train & Val (Murni)
X_train_raw, X_val, y_train_raw, y_val = train_test_split(X_samp, y_samp, test_size=0.2, stratify=y_samp, random_state=42)

# 3. Oversample HANYA kelas yang paling susah (Navigation & Service)
# Kita duplikasi 2x saja agar total baris tidak meledak
minoritas_mask = y_train_raw.isin(['Navigation_Issue', 'Service_Complaint'])
X_min = X_train_raw[minoritas_mask]
y_min = y_train_raw[minoritas_mask]

X_train_final = pd.concat([X_train_raw, X_min, X_min], axis=0)
y_train_final = pd.concat([y_train_raw, y_min, y_min], axis=0)

# 4. Hapus kolom yang terlalu unik/berat (High Cardinality) jika RAM mepet
# Kita simpan Brand, tapi hapus Pickup_Zone/Dropoff_Zone jika perlu. 
# Untuk sekarang kita coba biarkan dulu.

cat_features = X_train_final.select_dtypes(include=['object', 'category']).columns.tolist()
X_train_final[cat_features] = X_train_final[cat_features].fillna("MISSING").astype(str)
X_val[cat_features] = X_val[cat_features].fillna("MISSING").astype(str)
X_test_final[cat_features] = X_test_final[cat_features].fillna("MISSING").astype(str)

del X_train_raw, y_train_raw, X_min, y_min, X_samp, y_samp
gc.collect()
print(f"Cell 4 Baru: Train {len(X_train_final)} baris. Jauh lebih ringan!")

Cell 4 Baru: Train 896078 baris. Jauh lebih ringan!


In [5]:
model = CatBoostClassifier(
    iterations=800,       # Kurangi iterasi agar tidak terlalu lama
    learning_rate=0.08,   # Naikkan sedikit agar belajar lebih cepat
    depth=6,              # Turunkan ke 6 (Sangat signifikan mengurangi penggunaan RAM)
    task_type="CPU",
    thread_count=10,      # Sisakan 2 thread agar OS Ubuntu kamu tetap stabil
    loss_function='MultiClass',
    eval_metric='TotalF1',
    early_stopping_rounds=50,
    random_seed=42,
    verbose=50            # Print setiap 50 iterasi saja
)

# Training langsung tanpa buat objek Pool tambahan (untuk hemat RAM)
model.fit(
    X_train_final, y_train_final,
    cat_features=cat_features,
    eval_set=(X_val, y_val),
    use_best_model=True
)

0:	learn: 0.4112630	test: 0.6798759	best: 0.6798759 (0)	total: 2.92s	remaining: 38m 53s
50:	learn: 0.8216320	test: 0.6980356	best: 0.6990475 (14)	total: 2m 3s	remaining: 30m 10s
100:	learn: 0.8353410	test: 0.7009649	best: 0.7009649 (100)	total: 4m 23s	remaining: 30m 23s
150:	learn: 0.8394392	test: 0.7044416	best: 0.7044416 (150)	total: 6m 51s	remaining: 29m 28s
200:	learn: 0.8403876	test: 0.7045673	best: 0.7046147 (188)	total: 9m 20s	remaining: 27m 49s
250:	learn: 0.8406547	test: 0.7046347	best: 0.7047301 (214)	total: 11m 45s	remaining: 25m 43s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.7047300953
bestIteration = 214

Shrink model to first 215 iterations.


In [6]:
y_val_pred = model.predict(X_val).flatten()
print(f"LOKAL MACRO F1 (JUJUR): {f1_score(y_val, y_val_pred, average='macro'):.4f}")
print("-" * 30)
print(classification_report(y_val, y_val_pred))

LOKAL MACRO F1 (JUJUR): 0.5593
------------------------------
                   precision    recall  f1-score   support

 Fraud_Indication       0.99      0.97      0.98      8006
 Navigation_Issue       0.13      0.00      0.00     16036
     Perfect_Trip       0.72      0.99      0.84     87952
 Safety_Violation       0.99      0.95      0.97     32032
Service_Complaint       0.17      0.00      0.00     15974

         accuracy                           0.79    160000
        macro avg       0.60      0.58      0.56    160000
     weighted avg       0.68      0.79      0.70    160000



In [7]:
test_preds = model.predict(X_test_final).flatten()
submission = pd.DataFrame({"Trip_ID": df_test["Trip_ID"], "Trip_Label": test_preds})
submission.to_csv(os.path.join(SUB_PATH, "submission_jebol_06_v1.csv"), index=False)
print("Selesai! Bismillah 0.6+")

Selesai! Bismillah 0.6+
